# Session VII: Command Line Tools and Automation

Use Python to automate tasks and interact through the command line, including parsing arguments and running scripts from the terminal.

## Exercise 1: CLI Greeter and Calculator
**Objective:** Practice basic command-line parsing using a list that represents `sys.argv`.

Instead of reading real terminal input in the notebook, you will write a function that receives an argument list like:

```python
["greet", "Alice"]
["add", "3", "4"]
```

**Tasks:**
1. `run_command(argv)` should support two commands:
   - `greet <name>` → return `"Hello, <name>!"`
   - `add <x> <y>` → return the numeric sum as a string
2. Return a helpful usage message for missing or invalid input.
3. Use `try/except` to handle non-numeric values in `add`.


In [1]:
def run_command(argv):
    """Parse a simple command list and return a response string."""
    usage = "Usage: greet <name> | add <x> <y>"

    if not argv:
        return usage

    command = argv[0].lower()

    if command == "greet":
        if len(argv) != 2:
            return "Usage: greet <name>"
        return f"Hello, {argv[1]}!"

    if command == "add":
        if len(argv) != 3:
            return "Usage: add <x> <y>"
        try:
            x = float(argv[1])
            y = float(argv[2])
        except ValueError:
            return f"Error: '{argv[1]}' and '{argv[2]}' must both be numbers."
        return str(x + y)

    return f"Unknown command '{argv[0]}'. {usage}"


# --- TEST ---
print(run_command(["greet", "Alice"]))
# Expected: Hello, Alice!

print(run_command(["add", "3", "4"]))
# Expected: 7.0

print(run_command(["add", "three", "4"]))
# Expected: Error message about numeric input

# a few extra edge cases
print(run_command([]))
print(run_command(["greet"]))
print(run_command(["multiply", "2", "3"]))

Hello, Alice!
7.0
Error: 'three' and '4' must both be numbers.
Usage: greet <name> | add <x> <y>
Usage: greet <name>
Unknown command 'multiply'. Usage: greet <name> | add <x> <y>


---
## Exercise 2: Expense Lister with `argparse`
**Objective:** Build a real argument parser using optional flags.

You are given a list of expense records. Build a command-line style filter that can:
- show all expenses
- filter by category
- filter by a minimum amount

**Tasks:**
1. `build_parser()` → Create an `ArgumentParser` with:
   - `--category`
   - `--min` (float, default `0`)
2. `filter_expenses(expenses, category=None, min_amount=0)` → Return only matching expenses.
3. `format_expenses(expenses)` → Return a formatted multi-line string.


In [2]:
import argparse

expenses = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-02", "category": "Food", "amount": 34.20, "note": "Groceries"},
    {"date": "2026-10-03", "category": "Books", "amount": 45.00, "note": "Python book"},
]

def build_parser():
    """Return an argparse parser for expense filtering."""
    parser = argparse.ArgumentParser(description="List and filter expenses")
    parser.add_argument("--category", help="only show this category")
    parser.add_argument("--min", type=float, default=0, help="minimum amount")
    return parser

def filter_expenses(expenses, category=None, min_amount=0):
    """Return filtered expenses matching the given criteria."""
    result = []
    for exp in expenses:
        # case-insensitive so "food" and "Food" both work from the terminal
        if category and exp["category"].lower() != category.lower():
            continue
        if exp["amount"] < min_amount:
            continue
        result.append(exp)
    return result

def format_expenses(expenses):
    """Return a formatted text report for a list of expenses."""
    if not expenses:
        return "No expenses found."

    lines = [f"{'Date':<12}{'Category':<12}{'Amount':>8}  Note"]
    lines.append("-" * 44)
    for exp in expenses:
        lines.append(f"{exp['date']:<12}{exp['category']:<12}{exp['amount']:>8.2f}  {exp['note']}")
    lines.append("-" * 44)
    total = sum(exp["amount"] for exp in expenses)
    lines.append(f"{'Total':<24}{total:>8.2f}")
    return "\n".join(lines)


# --- TEST ---
parser = build_parser()
args = parser.parse_args(["--category", "Food", "--min", "20"])
filtered = filter_expenses(expenses, args.category, args.min)
print(format_expenses(filtered))

print()
# no flags -> everything
args = parser.parse_args([])
print(format_expenses(filter_expenses(expenses, args.category, args.min)))

Date        Category      Amount  Note
--------------------------------------------
2026-10-02  Food           34.20  Groceries
--------------------------------------------
Total                      34.20

Date        Category      Amount  Note
--------------------------------------------
2026-10-01  Food           18.50  Lunch
2026-10-02  Transport       2.80  Metro
2026-10-02  Food           34.20  Groceries
2026-10-03  Books          45.00  Python book
--------------------------------------------
Total                     100.50


---
## Exercise 3: Budget Backup Script
**Objective:** Automate data backups using `pathlib`, timestamps, and file copying.

Your script should validate that the source file exists, create a `backups` folder if needed, and make a timestamped copy.

**Tasks:**
1. `make_backup_name(filename)` → Return a new filename like `budget_20261005_224700.json`
2. `backup_file(source_path, backup_dir="backups")` → Copy the source file into the backup folder.
3. Return the destination path as a string, or `None` if the source file is missing.

**First, run the setup cell below to create a sample file.**


In [3]:
from pathlib import Path
import json

sample_budget = [
    {"category": "Food", "amount": 20.5},
    {"category": "Transport", "amount": 15.0},
]

with open("budget_data.json", "w") as file:
    json.dump(sample_budget, file, indent=2)

print("Created budget_data.json")


Created budget_data.json


In [4]:
from pathlib import Path
from datetime import datetime
import shutil

def make_backup_name(filename):
    """Return a timestamped backup filename."""
    path = Path(filename)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{path.stem}_{timestamp}{path.suffix}"

def backup_file(source_path, backup_dir="backups"):
    """Copy a file into backup_dir with a timestamped name."""
    source = Path(source_path)
    if not source.exists():
        print(f"Error: {source} does not exist, nothing to back up.")
        return None

    folder = Path(backup_dir)
    folder.mkdir(exist_ok=True)

    destination = folder / make_backup_name(source.name)
    shutil.copy2(source, destination)
    return str(destination)


# --- TEST ---
result = backup_file("budget_data.json")
print("Backup created:", result)

# missing file should return None instead of crashing
print("Missing file:", backup_file("does_not_exist.json"))

Backup created: backups/budget_data_20261112_145847.json
Error: does_not_exist.json does not exist, nothing to back up.
Missing file: None


---
## Exercise 4: CSV Expense Summary CLI
**Objective:** Load a CSV file, validate rows, and print category totals.

**Tasks:**
1. `load_expenses_csv(filename)` → Use `csv.DictReader` to load rows.
2. Skip malformed rows where:
   - `category` is empty
   - `amount` is missing or non-numeric
3. `summarize_by_category(expenses)` → Return a dictionary of totals.
4. `print_summary(summary)` → Print a formatted report.

**Run the setup cell below first.**


In [5]:
import csv

rows = [
    ["date", "category", "amount", "note"],
    ["2026-10-01", "Food", "18.50", "Lunch"],
    ["2026-10-02", "Transport", "2.80", "Metro"],
    ["2026-10-02", "", "12.00", "Broken row"],
    ["2026-10-03", "Books", "forty", "Invalid amount"],
    ["2026-10-04", "Food", "21.30", "Dinner"],
]

with open("expenses_cli.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("Created expenses_cli.csv")


Created expenses_cli.csv


In [6]:
import csv

def load_expenses_csv(filename):
    """Load valid expense rows from a CSV file."""
    valid = []
    try:
        with open(filename, "r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            # start=2 because line 1 is the header
            for line_no, row in enumerate(reader, start=2):
                category = (row.get("category") or "").strip()
                amount_raw = (row.get("amount") or "").strip()

                if not category:
                    print(f"Skipping line {line_no}: missing category")
                    continue
                try:
                    amount = float(amount_raw)
                except ValueError:
                    print(f"Skipping line {line_no}: invalid amount '{amount_raw}'")
                    continue

                row["category"] = category
                row["amount"] = amount
                valid.append(row)
    except FileNotFoundError:
        print(f"Error: {filename} not found.")
    return valid

def summarize_by_category(expenses):
    """Return {category: total_amount}."""
    totals = {}
    for exp in expenses:
        totals[exp["category"]] = totals.get(exp["category"], 0) + exp["amount"]
    return totals

def print_summary(summary):
    """Print a formatted category summary."""
    if not summary:
        print("No valid expenses to summarize.")
        return

    print("\nEXPENSE SUMMARY")
    print("-" * 26)
    # biggest category first
    for category, total in sorted(summary.items(), key=lambda item: item[1], reverse=True):
        print(f"{category:<15}{total:>10.2f}")
    print("-" * 26)
    print(f"{'Total':<15}{sum(summary.values()):>10.2f}")


# --- TEST ---
expenses = load_expenses_csv("expenses_cli.csv")
summary = summarize_by_category(expenses)
print_summary(summary)

Skipping line 4: missing category
Skipping line 5: invalid amount 'forty'

EXPENSE SUMMARY
--------------------------
Food                39.80
Transport            2.80
--------------------------
Total               42.60


---
## Exercise 5: Scheduling Helper
**Objective:** Generate schedule strings for cron and Windows Task Scheduler.

You are **not** actually scheduling tasks in this notebook. Instead, you will generate the correct command strings.

**Tasks:**
1. `build_cron_schedule(minute, hour, script_path)` → Return a cron line like:
   `30 18 * * * python /path/to/script.py`
2. `build_task_scheduler_command(time_str, script_path)` → Return a Windows-style command string.
3. Validate the time values and raise or handle `ValueError` for invalid input.


In [7]:
import re

def build_cron_schedule(minute, hour, script_path):
    """Return a cron schedule line for a daily script."""
    if not isinstance(minute, int) or not 0 <= minute <= 59:
        raise ValueError(f"minute must be 0-59, got {minute}")
    if not isinstance(hour, int) or not 0 <= hour <= 23:
        raise ValueError(f"hour must be 0-23, got {hour}")
    return f"{minute} {hour} * * * python {script_path}"

def build_task_scheduler_command(time_str, script_path):
    """Return a Windows schtasks command string."""
    # HH:MM, 00:00 to 23:59
    if not re.match(r"^([01]\d|2[0-3]):[0-5]\d$", time_str):
        raise ValueError(f"time must be in HH:MM format (00:00-23:59), got '{time_str}'")
    return f'schtasks /Create /SC DAILY /TN BudgetBackup /TR "python {script_path}" /ST {time_str}'


# --- TEST ---
print(build_cron_schedule(30, 18, "/home/user/backup.py"))
print(build_task_scheduler_command("18:30", "C:/scripts/backup.py"))

# invalid values should raise ValueError
for bad_call in [lambda: build_cron_schedule(75, 18, "backup.py"),
                 lambda: build_cron_schedule(0, 24, "backup.py"),
                 lambda: build_task_scheduler_command("6:30pm", "backup.py")]:
    try:
        bad_call()
    except ValueError as e:
        print("ValueError:", e)

30 18 * * * python /home/user/backup.py
schtasks /Create /SC DAILY /TN BudgetBackup /TR "python C:/scripts/backup.py" /ST 18:30
ValueError: minute must be 0-59, got 75
ValueError: hour must be 0-23, got 24
ValueError: time must be in HH:MM format (00:00-23:59), got '6:30pm'


---
## Exercise 6: Automation Suite
**Objective:** Combine parsing, loading JSON, listing expenses, backups, and summaries into one mini CLI application.

Build a tool with these subcommands:
- `list` → print all expenses
- `summary` → show totals by category
- `backup` → make a timestamped backup of the JSON file

**Tasks:**
1. `load_json_expenses(filename)` → Safely load a JSON list of expense dictionaries.
2. `build_suite_parser()` → Create an `argparse` parser with subcommands:
   - `list`
   - `summary`
   - `backup`
   Also add a common `filename` argument.
3. `run_suite(argv=None)` → Parse the arguments and perform the correct action.
4. Reuse functions from earlier exercises when possible.

**Run the setup cell below first.**


In [8]:
import json

suite_data = [
    {"date": "2026-10-01", "category": "Food", "amount": 18.50, "note": "Lunch"},
    {"date": "2026-10-02", "category": "Transport", "amount": 2.80, "note": "Metro"},
    {"date": "2026-10-03", "category": "Food", "amount": 21.30, "note": "Dinner"},
    {"date": "2026-10-04", "category": "Books", "amount": 45.00, "note": "Python book"},
]

with open("expenses_suite.json", "w") as file:
    json.dump(suite_data, file, indent=2)

print("Created expenses_suite.json")


Created expenses_suite.json


In [9]:
import argparse
import json

def load_json_expenses(filename):
    """Safely load and validate a JSON expense list."""
    try:
        with open(filename, "r", encoding="utf-8") as file:
            data = json.load(file)
    except FileNotFoundError:
        print(f"Error: {filename} not found.")
        return []
    except json.JSONDecodeError:
        print(f"Error: {filename} is not valid JSON.")
        return []

    if not isinstance(data, list):
        print("Error: JSON file must contain a list of expenses.")
        return []

    # keep only rows that have what we need
    required = {"date", "category", "amount"}
    return [row for row in data if isinstance(row, dict) and required.issubset(row)]

def build_suite_parser():
    """Return an argparse parser with subcommands."""
    parser = argparse.ArgumentParser(description="Expense automation suite")
    parser.add_argument("filename", help="path to the JSON expense file")

    subparsers = parser.add_subparsers(dest="command", required=True)
    subparsers.add_parser("list", help="show all expenses")
    subparsers.add_parser("summary", help="totals by category")
    subparsers.add_parser("backup", help="timestamped copy of the JSON file")
    return parser

def format_summary(summary):
    """Same layout as print_summary from Ex 4, but returns a string."""
    if not summary:
        return "No valid expenses to summarize."
    lines = ["EXPENSE SUMMARY", "-" * 26]
    for category, total in sorted(summary.items(), key=lambda item: item[1], reverse=True):
        lines.append(f"{category:<15}{total:>10.2f}")
    lines.append("-" * 26)
    lines.append(f"{'Total':<15}{sum(summary.values()):>10.2f}")
    return "\n".join(lines)

def run_suite(argv=None):
    """Run the automation suite and return a message or report."""
    parser = build_suite_parser()
    try:
        args = parser.parse_args(argv)
    except SystemExit:
        # argparse calls sys.exit on bad input, which would stop the notebook
        return "Usage: <filename> {list,summary,backup}"

    if args.command == "backup":
        result = backup_file(args.filename)          # from Ex 3
        if result is None:
            return "Backup failed."
        return f"Backup created: {result}"

    data = load_json_expenses(args.filename)
    if args.command == "list":
        return format_expenses(data)                 # from Ex 2
    if args.command == "summary":
        return format_summary(summarize_by_category(data))   # from Ex 4


# --- TEST ---
print(run_suite(["expenses_suite.json", "list"]))
print()
print(run_suite(["expenses_suite.json", "summary"]))
print()
print(run_suite(["expenses_suite.json", "backup"]))
print()
print(run_suite(["missing.json", "list"]))

Date        Category      Amount  Note
--------------------------------------------
2026-10-01  Food           18.50  Lunch
2026-10-02  Transport       2.80  Metro
2026-10-03  Food           21.30  Dinner
2026-10-04  Books          45.00  Python book
--------------------------------------------
Total                      87.60

EXPENSE SUMMARY
--------------------------
Books               45.00
Food                39.80
Transport            2.80
--------------------------
Total               87.60

Backup created: backups/expenses_suite_20261112_145847.json

Error: missing.json not found.
No expenses found.
